# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Convert metadata to dict for display
metadata = dataset.metadata.to_json()

print('Dataset Title:', metadata.get('name', ''))
print('Description:', metadata.get('description', ''))
print('Published:', metadata.get('datePublished', ''))
print('Identifier:', metadata.get('identifier', ''))

## 2. Data Overview
Review available record sets in the dataset, including their `@id`s and fields (`@id`s and labels).

In [ ]:
from pprint import pprint

# Get all record set metadata
record_sets = metadata.get('recordSet', [])
if isinstance(record_sets, dict):
    record_sets = [record_sets]

if not record_sets:
    print('No record sets found directly in metadata. Trying alternative keys...')
    # Sometimes Croissant puts recordSets under 'hasPart'
    record_sets = [x for x in metadata.get('hasPart', []) if x.get('@type', None) == 'RecordSet']
    if not record_sets:
        print('No record sets found.')

# Show each record set with fields
record_set_ids = []
for rs in record_sets:
    rs_id = rs.get('@id', None)
    rs_type = rs.get('@type', None)
    record_set_ids.append(rs_id)
    print(f'Record Set @id: {rs_id}, type: {rs_type}')
    # Fields
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(' Fields:')
    for f in fields:
        fid = f.get('@id', 'N/A')
        fname = f.get('name', f.get('label', ''))
        dtype = f.get('dataType', '')
        print(f'  - Field @id: {fid}, name: {fname}, type: {dtype}')
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above. If there are multiple record sets, this will load them all into separate DataFrames.

In [ ]:
# If there were no record sets found from metadata, manually specify the main record set `@id`.
# For this dataset, we inspect available data programmatically.
if not record_set_ids:
    # Try to infer a main record set (commonly 'http.../records/...')
    # Or list all possible sets via dataset._metadata['@graph']
    # Display a preview of the first 2 record sets found.
    print('Could not locate record set IDs to extract data.')
    print('Try calling dataset._metadata (advanced) to inspect available `@id`s.')
    graph = dataset._metadata.get('@graph', [])
    for obj in graph:
        if obj.get('@type') == 'RecordSet':
            print('Candidate RecordSet:', obj.get('@id'))
    # Let's pick a probable RecordSet id from the schema structure
    record_set_ids = [obj['@id'] for obj in graph if obj.get('@type') == 'RecordSet']

if not record_set_ids:
    raise RuntimeError('No RecordSet IDs found!')

dataframes = {}
for rsid in record_set_ids:
    try:
        records = list(dataset.records(record_set=rsid))
        print(f'Loaded {len(records)} records from RecordSet {rsid}')
        df = pd.DataFrame(records)
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(2))
        dataframes[rsid] = df
    except Exception as e:
        print(f'Failed to load from RecordSet {rsid}: {e}')

# Set the main record set id for the rest of the notebook
main_record_set_id = record_set_ids[0]
print(f\nMain Record Set chosen for EDA: {main_record_set_id}')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes filtering, normalization, and grouping by key attributes.

In [ ]:
# Choose a numeric field by inspecting dataframe columns
df = dataframes[main_record_set_id]
print('Columns available in main record set:')
print(df.columns.tolist())
# Let's pick a likely quantitative field, e.g. 'coverage' or 'MW' or a peptide count
numeric_fields = [c for c in df.columns if df[c].dtype.kind in 'fi' and df[c].notnull().any()]
if not numeric_fields:
    # Alternatively, try columns expected to be numeric (by name)
    for fallback in ['coverage', 'MW', 'psms', 'unique_peptides', 'abundance']:
        if fallback in df.columns:
            numeric_fields.append(fallback)
if not numeric_fields:
    raise ValueError('No numeric fields found. Please inspect columns in the previous cell.')
numeric_field = numeric_fields[0]
print(f'Using numeric_field = {numeric_field}')

# Filter records where numeric_field > threshold
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f} (count = {filtered_df.shape[0]})")
print(filtered_df.head(3))

# Normalize the numeric field
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by another field, e.g. 'description' or 'accession' or any categorical column with few unique values
group_fields = [c for c in df.columns if df[c].dtype == object and df[c].nunique() < 20]
group_field = group_fields[0] if group_fields else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"\nGrouped by {group_field} (top 5):")
    print(grouped_df.head())
else:
    print('No suitable categorical group_field found.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the numeric field (histogram)
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If a group field was found, visualize mean values
if group_field:
    plt.figure(figsize=(8,4))
    group_means = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
    group_means.plot(kind='bar')
    plt.title(f'Mean {numeric_field} grouped by {group_field}')
    plt.ylabel(f'Mean {numeric_field}')
    plt.xlabel(group_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR² dataset using the Croissant schema via the `mlcroissant` library. After loading the dataset and extracting records from the main record set (referenced by its `@id`), we:

- Inspected the available record sets and their fields by `@id`.
- Loaded all records into Pandas DataFrames for analysis.
- Filtered and normalized a selected numeric field.
- Optionally grouped and visualized the data by categorical variables.
- Plotted value distributions to explore the data visually.

Please refer to the schema documentation for details about the field semantics. For advanced analysis, you can further join subsets, calculate correlations, or engineer new features as required.